In [1]:
from multiprocessing import Process, Manager, Event
from threading import Thread
import pandas as pd
import traceback
import websocket
import json
import datetime
import time
# set directory to upper directory
import os
import sys
# sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname(__file__))))
sys.path.append("/home/kp_info_loader")
from loggers.logger import KimpBotLogger
from exchange_websocket.utils import list_slice

# temporary
from etc.register_monitor_msg import RegisterMonitorMsg

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [2]:
with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
        config = json.load(f)
node = config['node']
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id_list = []
admin_id = config['telegram_admin_id']['charlie1155']
admin_id_list.append(admin_id)
logging_dir = None
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
class OkxWebsocket:
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, logging_dir=None):
        self.url = "wss://ws.okx.com:8443/ws/v5/public"
        self.market_type = market_type
        self.admin_id = admin_id
        self.node = node
        self.register_monitor_msg = register_monitor_msg
        self.get_symbol_list = get_symbol_list
        self.websocket_logger = KimpBotLogger(f"okx_{self.market_type.lower()}_websocket", logging_dir).logger
        manager = Manager()
        self.ticker_dict = manager.dict()
        self.proc_n = proc_n
        self.before_symbols_list = self.get_symbol_list()
        self.sliced_symbols_list = list_slice(self.get_symbol_list(), self.proc_n)
        self.stop_restart_webscoket = False
        self.price_proc_event_list = []
        self.websocket_proc_dict = {}
        self.websocket_symbol_dict = {}
        self._start_websocket()
        self.monitor_shared_symbol_change_thread = Thread(target=self.monitor_shared_symbol_change, daemon=True)
        self.monitor_shared_symbol_change_thread.start()
        self.monitor_websocket_last_update_thread = Thread(target=self.monitor_websocket_last_update, daemon=True)
        self.monitor_websocket_last_update_thread.start()

    def init_websocket(self, ticker_dict, data, error_event):
        def on_message(ws, message):
            if error_event.is_set():
                ws.close()
                raise Exception("okx_websocket|error_event is set. closing websocket..")
            message_dict = json.loads(message)
            # print(message_dict)
            if 'data' in message_dict.keys():
                self.ticker_dict[message_dict['data'][0]['instId']] = {**message_dict['data'][0], "last_update": datetime.datetime.now()}

        def on_error(ws, error):
            # print(f'okx_websocket on_error executed!')
            # print(error)
            self.websocket_logger.error(f'okx_websocket|okx_websocket on_error executed!\n Error: {error}')
            pass

        def on_close(ws, close_status_code, close_msg):
            print(f"\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")
            # self.websocket_logger.info(f"okx_websocket|\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")

        def on_open(ws):
            print(f'okx_websocket started')
            # self.websocket_logger.info(f'okx_websocket|okx_websocket started')
            ws.send(json.dumps(data))

        websocket.enableTrace(False)
        ws = websocket.WebSocketApp(self.url,
                                on_open=on_open,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close)
        ws.run_forever(ping_interval=15)
    
    def _start_websocket(self):
        def handle_price_procs():
            while True:
                try:
                    if self.stop_restart_webscoket is False:
                        for i in range(self.proc_n):
                            ticker_start_proc = False
                            ticker_restarted = False
                            if f"{i+1}th_ticker_proc" in self.websocket_proc_dict.keys() and not self.websocket_proc_dict[f"{i+1}th_ticker_proc"].is_alive():
                                ticker_start_proc = True
                                ticker_restarted = True
                                self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                                self.websocket_logger.info(f"okx_ticker_websocket|{i+1}th okx_ticker_proc terminated.")
                            elif f"{i+1}th_ticker_proc" not in self.websocket_proc_dict.keys():
                                ticker_start_proc = True
                                self.websocket_logger.info(f"{i+1}th Okx ticker websocket does not exist. starting..")
                                self.websocket_logger.info(f"okx_ticker_websocket|{i+1}th okx_ticker_proc started.")
                            if ticker_start_proc is True:
                                error_event = Event()
                                self.price_proc_event_list.append(error_event)
                                self.websocket_symbol_dict[f"{i+1}th_ticker_symbol"] = self.sliced_symbols_list[i]
                                ticker_data = {
                                    "op": "subscribe",
                                    "args": [
                                        {"channel": "tickers", "instId": f"{x}"} for x in self.sliced_symbols_list[i]
                                    ]
                                }
                                ticker_proc = Process(target=self.init_websocket, args=(self.ticker_dict, ticker_data, error_event), daemon=True)
                                self.websocket_proc_dict[f"{i+1}th_ticker_proc"] = ticker_proc
                                ticker_proc.start()
                                if ticker_restarted:
                                    content = f"restarted {i+1}th ticker websocket.. alive state: {self.websocket_proc_dict[f'{i+1}th_ticker_proc'].is_alive()}"
                                    self.websocket_logger.info(f"ticker_websocket|{content}")
                                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', f'OKX {self.market_type} ticker websocket restart', content, code=None, sent_switch=0, send_counts=1, remark=None)
                            time.sleep(0.5)
                except Exception as e:
                    content = f"handle_price_procs|{traceback.format_exc()}"
                    self.websocket_logger.error(content)
                    self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"[OKX {self.market_type}]handle_price_procs", content=content, code=None, sent_switch=0, send_counts=1, remark=None)
                    time.sleep(1)
                time.sleep(0.5)
        self.handle_price_procs_thread = Thread(target=handle_price_procs, daemon=True)
        self.handle_price_procs_thread.start()

    def terminate_websocket(self):
        self.stop_restart_webscoket = True
        time.sleep(0.5)
        for each_event in self.price_proc_event_list:
            each_event.set()
        self.websocket_logger.info(f"[OKX {self.market_type}]all websockets' event has been set")
        self.price_proc_event_list = []
        time.sleep(1)
        self.stop_restart_webscoket = False
    
    def check_status(self, print_result=False, include_text=False):
        if len(self.websocket_proc_dict) == 0:
            proc_status = False
            print_text = f"[OKX {self.market_type}]websocket proc is not running."
            if print_result:
                print(print_text)
            if include_text:
                return (proc_status, print_text)
            return proc_status
        else:
            proc_status = all([x.is_alive() for x in self.websocket_proc_dict.values()])
            print_text = ""
            for key, value in self.websocket_proc_dict.items():
                print_text += f"[OKX {self.market_type}]{key} status: {value.is_alive()}\n"
            if print_result:
                print(print_text)
            if include_text:
                return (proc_status, print_text)
            return proc_status

    def monitor_shared_symbol_change(self, loop_time_secs=60):
        self.websocket_logger.info(f"[OKX {self.market_type}]started monitor_shared_symbol_change..")
        while True:
            time.sleep(loop_time_secs)
            try:
                restart_websockets = False
                new_symbols_list = self.get_symbol_list()
                
                if sorted(self.before_symbols_list) != sorted(new_symbols_list):
                    restart_websockets = True
                    deleted_spot_shared_symbol = [x for x in self.before_symbols_list if x not in new_symbols_list]
                    added_spot_shared_symbol = [x for x in new_symbols_list if x not in self.before_symbols_list]
                    content = f"monitor_shared_symbol_change|[OKX {self.market_type}]shared symbol changed. deleted: {deleted_spot_shared_symbol}, added: {added_spot_shared_symbol}"
                    self.websocket_logger.info(content)
                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_shared_symbol_change', content, code=None, sent_switch=0, send_counts=1, remark=None)
                    for each_shared_symbol in deleted_spot_shared_symbol:
                        # remove deleted symbol from ticker_dict
                        try:
                            del self.ticker_dict[each_shared_symbol]
                        except Exception:
                            pass
                
                if restart_websockets is True:
                    # Set the newer values to before values
                    self.before_symbols_list = new_symbols_list
                    # Set sliced values too
                    self.sliced_symbols_list = list_slice(self.get_symbol_list(), self.proc_n)
                    # terminate websockets
                    self.terminate_websocket()
            except Exception as e:
                content = f"monitor_shared_symbol_change|{traceback.format_exc()}"
                self.websocket_logger.error(content)
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"monitor_shared_symbol_change", content=content, code=None, sent_switch=0, send_counts=1, remark=None)

    def monitor_websocket_last_update(self, update_threshold_mins=10, loop_time_secs=15):
        self.websocket_logger.info(f"[OKX {self.market_type}]started monitor_websocket_last_update..")
        while True:
            time.sleep(loop_time_secs)
            try:
                ticker_df = pd.DataFrame(dict(self.ticker_dict)).T.reset_index()
                for i in range(self.proc_n):
                    allocated_symbol_list = self.websocket_symbol_dict[f"{i+1}th_ticker_symbol"]
                    # check ticker dict's last_update
                    allocated_ticker_df = ticker_df[ticker_df['instId'].isin(allocated_symbol_list)]
                    if len(allocated_ticker_df) == 0:
                        content = f"monitor_websocket_last_update|{i+1}th_ticker_proc has no ticker_dict data. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                        continue
                    ticker_last_update = allocated_ticker_df['last_update'].max()
                    # check orderbook dict's last_update
                    # If the last update is older than update_threshold_mins, restart websocket
                    if (datetime.datetime.now() - ticker_last_update).total_seconds() / 60 > update_threshold_mins:
                        content = f"monitor_websocket_last_update|{i+1}th_ticker_proc last_update is older than {update_threshold_mins} mins. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
            except Exception as e:
                content = f"monitor_websocket_last_update|{traceback.format_exc()}"
                self.websocket_logger.error(content)
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"[OKX {self.market_type}] monitor_websocket_last_update", content=content, code=None, sent_switch=0, send_counts=1, remark=None)

    def get_price_df(self):
        ticker_df = pd.DataFrame(self.ticker_dict.values())
        ticker_df['base_asset'] = ticker_df['instId'].apply(lambda x: x.split('-')[0])
        ticker_df['quote_asset'] = ticker_df['instId'].apply(lambda x: x.split('-')[1])
        ticker_df = ticker_df.rename(columns={"last": "tp", "askPx": "ap", "bidPx":"bp", "volCcy24h":"atp24h"})
        ticker_df.loc[:, ['tp', 'ap', 'bp', 'open24h', 'atp24h']] = ticker_df.loc[:, ['tp', 'ap', 'bp', 'open24h', 'atp24h']].astype(float)
        ticker_df['atp24h'] = ticker_df.apply(lambda x: x['tp']*x['atp24h'] if x['instType'] != "SPOT" else x['atp24h'], axis=1)
        ticker_df['scr'] = (ticker_df['tp'] - ticker_df['open24h'])/ticker_df['open24h'] * 100
        ticker_df = ticker_df[['instId', 'base_asset', 'quote_asset', 'tp', 'ap', 'bp', 'scr', 'atp24h', 'last_update']]
        return ticker_df

class OkxUSDMWebsocket(OkxWebsocket):
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, logging_dir):
        super().__init__(admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, logging_dir)

class OkxCOINMWebsocket(OkxWebsocket):
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, logging_dir):
        super().__init__(admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, logging_dir)
        


In [3]:
node = 'test'
proc_n = 1
get_symbol_list_spot = lambda: ["BTC-USDT"]
get_symbol_list_usd_m = lambda: ["BTC-USDT-SWAP"]
get_symbol_list_coin_m = lambda: ["BTC-USD-SWAP"]
okx_websocket = OkxWebsocket(admin_id, node, proc_n, get_symbol_list_spot, register_monitor_msg, "SPOT", logging_dir)
okx_usd_m_websocket = OkxUSDMWebsocket(admin_id, node, proc_n, get_symbol_list_usd_m, register_monitor_msg, "USD_M", logging_dir)
okx_coin_m_websocket = OkxCOINMWebsocket(admin_id, node, proc_n, get_symbol_list_coin_m, register_monitor_msg, "COIN_M", logging_dir)

2023-10-31 15:43:30,798 - okx_spot_websocket - INFO - 1th Okx ticker websocket does not exist. starting..
2023-10-31 15:43:30,800 - okx_spot_websocket - INFO - [OKX SPOT]started monitor_shared_symbol_change..
2023-10-31 15:43:30,801 - okx_spot_websocket - INFO - okx_ticker_websocket|1th okx_ticker_proc started.
2023-10-31 15:43:30,801 - okx_spot_websocket - INFO - [OKX SPOT]started monitor_websocket_last_update..
2023-10-31 15:43:30,819 - okx_usd_m_websocket - INFO - 1th Okx ticker websocket does not exist. starting..
2023-10-31 15:43:30,820 - okx_usd_m_websocket - INFO - [OKX USD_M]started monitor_shared_symbol_change..
2023-10-31 15:43:30,821 - okx_usd_m_websocket - INFO - [OKX USD_M]started monitor_websocket_last_update..
2023-10-31 15:43:30,822 - okx_usd_m_websocket - INFO - okx_ticker_websocket|1th okx_ticker_proc started.
2023-10-31 15:43:30,835 - okx_coin_m_websocket - INFO - 1th Okx ticker websocket does not exist. starting..
2023-10-31 15:43:30,836 - okx_coin_m_websocket - INF

okx_websocket started
okx_websocket started
okx_websocket started


In [4]:
okx_websocket.check_status(print_result=True, include_text=True)

[OKX SPOT]1th_ticker_proc status: True



(True, '[OKX SPOT]1th_ticker_proc status: True\n')

In [6]:
pd.DataFrame(okx_websocket.ticker_dict.values())

,instType,instId,last,lastSz,askPx,askSz,bidPx,bidSz,open24h,high24h,low24h,sodUtc0,sodUtc8,volCcy24h,vol24h,ts,last_update
0,SPOT,BTC-USDT,34309.4,0.00314965,34309.4,1.23312267,34309.3,0.89159101,34256.3,34868,34016.5,34480.4,34656.5,385095312.632227586,11176.10351068,1698731476608,2023-10-31 14:51:16.670017
1,SWAP,BTC-USDT-SWAP,34307.3,15,34307.3,504,34307.2,746,34255.2,34882.2,34004.7,34476.6,34660.4,122469.56,12246956,1698731477009,2023-10-31 14:51:17.070944
2,SWAP,BTC-USD-SWAP,34318.6,26,34318.6,324,34318.5,920,34265,34888,34040.9,34500.1,34674.9,13554.7401,4673209,1698731477024,2023-10-31 14:51:17.092740


In [5]:
okx_websocket.get_price_df()

,instId,base_asset,quote_asset,tp,ap,bp,scr,atp24h,last_update
0,BTC-USDT,BTC,USDT,34308.7,34308.8,34308.7,0.048408,3.880022e+08,2023-10-31 15:43:45.070112


In [6]:
okx_usd_m_websocket.get_price_df()

,instId,base_asset,quote_asset,tp,ap,bp,scr,atp24h,last_update
0,BTC-USDT-SWAP,BTC,USDT,34310.2,34310.2,34310.1,0.051323,4.242067e+09,2023-10-31 15:43:50.681136


In [8]:
okx_coin_m_websocket.get_price_df()

,instId,base_asset,quote_asset,tp,ap,bp,scr,atp24h,last_update
0,BTC-USD-SWAP,BTC,USD,34322.2,34320.1,34320.0,0.069683,4.714898e+08,2023-10-31 15:43:58.026676
